In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# ===== 1. Khai báo đường dẫn file input =====
# Đổi lại 2 đường dẫn này theo máy của bạn
sentiment_path = Path(r"D:\User\Tài liệu học\CCDNCTKHDL\5. Sentiment score\Sentiment_Score_LLM.csv")
satisfaction_path = Path(r"D:\User\Tài liệu học\CCDNCTKHDL\4. Satisfaction_score\Clause_Final_with_Satisfaction_Score.csv")

# Giữ nguyên tên output như file Matrix.ipynb cũ
output_path = sentiment_path.parent / "Matrix_wide.csv"

In [3]:
# ===== 2. Đọc file =====
df_sentiment = pd.read_csv(sentiment_path)
df_satisfaction = pd.read_csv(satisfaction_path)

print("Sentiment file:", df_sentiment.shape)
print("Satisfaction file:", df_satisfaction.shape)
print("Output path:", output_path)

Sentiment file: (231726, 10)
Satisfaction file: (231726, 10)
Output path: D:\User\Tài liệu học\CCDNCTKHDL\5. Sentiment score\Matrix_wide.csv


In [4]:
# ===== 3. Chuẩn hóa cột điểm về dạng số =====
df_sentiment["senti_score"] = pd.to_numeric(df_sentiment["senti_score"], errors="coerce").fillna(0)

df_satisfaction["Satisfaction_Score"] = pd.to_numeric(
    df_satisfaction["Satisfaction_Score"],
    errors="coerce"
)

In [5]:
# ===== 4. Tạo satisfaction_score: gom theo review_id và lấy trung bình =====
df_satisfaction_review = (
    df_satisfaction
    .groupby("review_id", as_index=False)["Satisfaction_Score"]
    .mean()
    .rename(columns={"Satisfaction_Score": "satisfaction_score"})
)

print("Số review có satisfaction_score:", df_satisfaction_review["review_id"].nunique())
print(df_satisfaction_review.head())

Số review có satisfaction_score: 64406
   review_id  satisfaction_score
0          1            0.724000
1          2            0.710000
2          3            0.410000
3          4            0.582000
4          5            0.516667


In [6]:
# ===== 5. Tạo sentiment_score dạng wide theo Label/topic =====
df_sentiment_wide = (
    df_sentiment.pivot_table(
        index="review_id",       # mỗi review là 1 dòng
        columns="Label",         # mỗi Label/topic là 1 cột
        values="senti_score",    # điểm sentiment
        aggfunc="sum",           # cộng điểm các clause cùng Label trong 1 review
        fill_value=0
    )
    .reset_index()
)

df_sentiment_wide.columns.name = None

print("Số review trước khi xử lý Outlier:", len(df_sentiment_wide))
print("Số cột trước khi xử lý Outlier:", df_sentiment_wide.shape[1])
print("Các cột hiện có:", list(df_sentiment_wide.columns))

Số review trước khi xử lý Outlier: 64406
Số cột trước khi xử lý Outlier: 17
Các cột hiện có: ['review_id', 'AR', 'AT', 'CS', 'GS', 'HH', 'LV', 'MH', 'NQ', 'OA', 'Outlier', 'PE', 'RC', 'RE', 'RP', 'SR', 'VP']


In [7]:
# ===== 6. Xóa cột Outlier / Outliner nếu có =====
outlier_cols = [
    col for col in df_sentiment_wide.columns
    if str(col).strip().lower() in ["outlier", "outliner", "-1"]
]

df_sentiment_wide = df_sentiment_wide.drop(columns=outlier_cols, errors="ignore")

print("Xóa cột Outlier/Outliner")
print("Đã xóa cột:", outlier_cols)
print("Còn lại số cột:", df_sentiment_wide.shape[1])

Xóa cột Outlier/Outliner
Đã xóa cột: ['Outlier']
Còn lại số cột: 16


In [8]:
# ===== 7. Xóa review không còn topic có ý nghĩa =====
# Sau khi bỏ Outlier, nếu toàn bộ điểm ở các topic đều bằng 0 thì xóa dòng đó.
topic_cols = [col for col in df_sentiment_wide.columns if col != "review_id"]

rows_before = len(df_sentiment_wide)
all_topic_zero_mask = (df_sentiment_wide[topic_cols].fillna(0) == 0).all(axis=1)
removed_rows = int(all_topic_zero_mask.sum())

df_sentiment_wide = df_sentiment_wide.loc[~all_topic_zero_mask].copy()

print("Xóa review toàn 0 sau khi bỏ Outlier")
print("Đã xóa dòng:", removed_rows)
print("Còn lại dòng:", len(df_sentiment_wide))

Xóa review toàn 0 sau khi bỏ Outlier
Đã xóa dòng: 3914
Còn lại dòng: 60492


In [9]:
# ===== 8. Ghép satisfaction_score vào matrix sentiment =====
df_matrix = df_sentiment_wide.merge(
    df_satisfaction_review,
    on="review_id",
    how="left"
)

missing_satisfaction = int(df_matrix["satisfaction_score"].isna().sum())

print("Ghép satisfaction_score")
print("Dòng thiếu satisfaction_score:", missing_satisfaction)
print("Tổng dòng matrix cuối:", len(df_matrix))

Ghép satisfaction_score
Dòng thiếu satisfaction_score: 0
Tổng dòng matrix cuối: 60492


In [10]:
# ===== 9. Sắp xếp cột cho dễ nhìn =====
# Nếu muốn đổi thứ tự topic, bạn chỉ cần sửa list này.
desired_topic_order = [
    "AR", "AT", "CS", "GS", "HH", "LV", "MH", "NQ",
    "OA", "PE", "RC", "RE", "RP", "SR", "VP"
]

existing_topic_order = [col for col in desired_topic_order if col in df_matrix.columns]

other_cols = [
    col for col in df_matrix.columns
    if col not in ["review_id", "satisfaction_score"] + existing_topic_order
]

df_matrix = df_matrix[
    ["review_id"] + existing_topic_order + other_cols + ["satisfaction_score"]
]

print("Số dòng cuối:", df_matrix.shape[0])
print("Số cột cuối:", df_matrix.shape[1])
print(df_matrix.head())

Số dòng cuối: 60492
Số cột cuối: 17
   review_id   AR   AT   CS   GS   HH   LV   MH        NQ   OA   PE       RC  \
0          1  0.0  0.0  0.0  0.0  0.0  0.0  0.0  2.424682  0.0  0.0  0.00000   
1          2  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.000000  0.0  0.0  0.00000   
2          3  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.944161  0.0  0.0  0.00000   
3          4  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.000000  0.0  0.0  0.91022   
4          5  0.0  0.0  0.0  0.0  0.0  0.0  0.0 -0.817978  0.0  0.0  0.00000   

    RE   RP   SR        VP  satisfaction_score  
0  0.0  0.0  0.0  0.962228            0.724000  
1  0.0  0.0  0.0  0.528349            0.710000  
2  0.0  0.0  0.0 -0.312881            0.410000  
3  0.0  0.0  0.0  1.689979            0.582000  
4  0.0  0.0  0.0  0.133833            0.516667  


In [11]:
# ===== 10. Lưu file về máy =====
# Vì bạn chạy trên VSCode nên dòng này sẽ lưu trực tiếp vào máy,
# cùng thư mục với file Sentiment_Score_LLM.csv.
df_matrix.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Đã lưu file:", output_path)
print("Số dòng đã lưu:", df_matrix.shape[0])
print("Số cột đã lưu:", df_matrix.shape[1])

Đã lưu file: D:\User\Tài liệu học\CCDNCTKHDL\5. Sentiment score\Matrix_wide.csv
Số dòng đã lưu: 60492
Số cột đã lưu: 17
